# ЛАБОРАТОРНА РОБОТА  3

In [25]:
import numpy as np
import math
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [26]:
N = 200    
K = 50           

## Метод оптимізації зграєю зозуль
- p_detect - ймовірність знаходження яйця 
- δ - параметр для генерування нового положення δ ∈ (0; 1)


### $$ x_{j}^{cur} = x_{k,j} + \delta \cdot (x_{j}^{max} - x_{j}^{min}) \cdot (-1 + 2 \cdot rand()) $$

In [27]:
def cuckoo_method(F, dim, x_min, x_max, p_detect, delta):
  
    population_history = []
    best_history = []
    
    x_best = x_min + (x_max - x_min) * np.random.rand(dim)

    P = []
    
    k = 0
    while k < K:
        
        x_k = x_min + (x_max - x_min) * np.random.rand(dim)
        P.append(x_k)
        k += 1

    P = np.array(P)

    for _ in range(N):

        population_history.append(P.copy())
        best_history.append(F(x_best)) 

        k = int(round(1 + (K - 1) * np.random.rand())) - 1

        x_cur = P[k] + delta * (x_max - x_min) * (-1 + 2 * np.random.rand(dim))

        x_cur = np.maximum(x_min, x_cur)
        x_cur = np.minimum(x_max, x_cur)

        if F(x_cur) < F(x_best):
            x_best = x_cur.copy()
            P[k] = x_cur.copy()

        if np.random.rand() >= p_detect:
            continue

        fitness = np.array([F(x) for x in P])
        m = np.argmax(fitness)

        x_m = P[m] + delta * (x_max - x_min) * (-1 + 2 * np.random.rand(dim))

        x_m = np.maximum(x_min, x_m)
        x_m = np.minimum(x_max, x_m)

        P[m] = x_m

    return x_best, F(x_best), population_history, best_history

## Оптимізація методом зграї кажанів
- r0 - початкова частота випромінювання 
- A0 - початкова амплітуда  
- α -  параметр для обчислення амплітуди
- γ -  параметр для обчислення радіуса
- δ -  параметр для генерації нового положення
- f_max, f_min - межі для частоти 

In [28]:
def bat_algorithm(F, dim, x_min, x_max, f_min, f_max, A0, r0, alpha, gamma, delta):

    population_history = []
    best_history = []

    P = np.random.uniform(x_min, x_max, (K, dim))
    V = np.zeros((K, dim))                         
    f = np.full(K, f_min)      
    
    A = np.full(K, A0)   
    r = np.full(K, r0)                   

    fitness = np.array([F(x) for x in P])
    best_idx = np.argmin(fitness)
    x_best = P[best_idx].copy()

    for n in range(1, N + 1):

        population_history.append(P)
        best_history.append(F(x_best)) 

        for k in range(K):

            f[k] = f_min + (f_max - f_min) * np.random.rand()

            V[k] = V[k] + (x_best - P[k]) * f[k]

            P[k] = P[k] + V[k]

            P[k] = np.maximum(x_min, P[k])
            P[k] = np.minimum(x_max, P[k])

            if np.random.rand() < r[k]:

                x_cur = x_best + delta * (x_max - x_min) * (-1 + 2 * np.random.rand(dim))

                x_cur = np.maximum(x_min, x_cur)
                x_cur = np.minimum(x_max, x_cur)
                
                if (F(x_cur) <= F(P[k])) and (np.random.rand() < A[k]):
                    
                    P[k] = x_cur.copy()

                    A[k] = alpha * A[k]
                    r[k] = r0 * (1 - np.exp(-gamma * n))

                    if F(x_cur) <= F(x_best):
                        x_best = x_cur.copy()

        fitness = np.array([F(x) for x in P])
        best_idx = np.argmin(fitness)
        if fitness[best_idx] < F(x_best):
            x_best = P[best_idx].copy()

    return x_best, F(x_best), population_history, best_history

## Функція для візуалізації 

In [29]:
def create_gif(population_history, best_history, f, name, filename, dim, x_bounds=None, y_bounds=None):

    if dim == 2:
        fig = plt.figure(figsize=(14, 10))

        ax3d = fig.add_subplot(2, 2, 1, projection='3d')
        ax_contour = fig.add_subplot(2, 2, 2)
        ax_hist = fig.add_subplot(2, 2, 3)
        ax_conv = fig.add_subplot(2, 2, 4)

        X = np.linspace(*x_bounds, 80)
        Y = np.linspace(*y_bounds, 80)
        X, Y = np.meshgrid(X, Y)

        Z = np.array([
            f(np.array([x, y]))
            for x, y in zip(X.ravel(), Y.ravel())
        ]).reshape(X.shape)

    else:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        ax_hist, ax_conv = axes
        
    suptitle = fig.suptitle("", fontsize=16, fontweight="bold")
    best_val = best_history[-1]
    
    ymin = np.min(best_history)
    ymax = np.max(best_history)
    
    frame_indices = list(range(0, len(population_history), 5))

    def update(frame_idx):
        
        frame = frame_indices[frame_idx]
        
        suptitle.set_text( f"{name}:" f" Найкраще знайдене значення: {best_val:.6f}\n"
                            f"Покоління: {frame + 1} / {len(population_history)}")

        if dim == 2:
            ax3d.cla()
            ax_contour.cla()
            
        ax_hist.cla()
        ax_conv.cla()

        pop = population_history[frame]
        fitness = np.array([f(ind) for ind in pop])

        if dim == 2:
            
            pop_x = pop[:, 0]
            pop_y = pop[:, 1]
            pop_z = np.array([f(ind) for ind in pop])
            
            ax3d.plot_surface(X, Y, Z, alpha=0.6)
            ax3d.scatter(pop_x, pop_y, pop_z, color='red')
            ax3d.set_title(f"3D Графік ")
            ax3d.set_xlim(x_bounds)
            ax3d.set_ylim(y_bounds)

            ax_contour.contourf(X, Y, Z, levels=50)
            ax_contour.scatter(pop_x, pop_y, color='red')
            ax_contour.set_title("Контурний графік")
            ax_contour.set_xlim(x_bounds)
            ax_contour.set_ylim(y_bounds)

        ax_hist.hist(fitness, bins=15)
        ax_hist.set_title("Графік пристосованості")
        ax_conv.plot(best_history[:frame + 1], lw=2)
        ax_conv.set_ylim(ymin, ymax)
        ax_conv.set_title("Графік збіжності")
        ax_conv.set_xlabel("Ітерація")
        ax_conv.set_ylabel("Best f(x)")
        ax_conv.set_xlim(0, len(population_history))
        ax_conv.grid(True)

        if dim == 2:
            return ax3d, ax_contour, ax_hist, ax_conv
        else:
            return ax_hist, ax_conv

    ani = animation.FuncAnimation(
        fig,
        update,
        frames = len(frame_indices),
        interval = 500,
        repeat = False
    )
    ani.save(filename, writer="pillow")
    plt.close(fig)

## 1. Протестувати кожен з розглянутих методів на функціях

In [30]:
def f_rastrigin(v, A=10):
    x = np.array(v)
    n = len(x)
    return A*n + np.sum(x**2 - A*np.cos(2*np.pi*x))

def mishra_bird(x):
    term1 = np.exp((1 - np.cos(x[0]))**2) * np.sin(x[1])
    term2 = np.exp((1 - np.sin(x[1]))**2) * np.cos(x[0])
    term3 = (x[0] - x[1])**2
    return term1 + term2 + term3

def constraint_mishra_bird(x):
    return (25 - (x[0] + 5)**2 - (x[1] + 5)**2) > 0

In [31]:
def make_penalty_function(func, constraint):

    def F_penalty(x):
        if constraint(x):
            return func(x)          
        else:
            return func(x) + 1000   

    return F_penalty

In [32]:
FUNCTIONS = {
    
    "rastrigin": {
        "func": f_rastrigin,
        "bounds": [(-5.12, 5.12)] * 20,
        "constraint": lambda x: True,
        "name": "Rastrigin function",
        "dim" : 20
    },

    "mishra_bird": {
        "func": mishra_bird,
        "bounds": [(-10, 0), (-6.5, 0)],
        "constraint" : constraint_mishra_bird ,
        "name": "Mishra-Bird function",
        "dim" : 2
    }
}

methods = {
    "cuckoo": cuckoo_method,
    "bat": bat_algorithm
}

In [33]:
for func_key, data in FUNCTIONS.items():

    func = data["func"]
    constraint = data["constraint"]
    bounds = data["bounds"]
    name = data["name"]
    dim = data["dim"]

    F_penalty = make_penalty_function(func, constraint)

    x_min = np.array([b[0] for b in bounds])
    x_max = np.array([b[1] for b in bounds])
    
    for method_name, method in methods.items():

        if method_name == "cuckoo":   
            best_x, best_f, pop_history, best_history = method( F_penalty, dim,x_min, x_max, p_detect=0.25, delta=0.5)

        elif method_name == "bat": 
            best_x, best_f, pop_hist, best_hist = method(F_penalty, dim, x_min, x_max, f_min=0.0, f_max=2.0, A0=1.5, r0=0.5, alpha=0.9, gamma=0.9, delta=0.5 )
            
        create_gif(
            pop_history,
            best_history,
            F_penalty,
            f"{method_name} - {name}",
            f"function/{method_name}_{func_key}.gif",
            dim,
            bounds[0],
            bounds[1]
        )